In [1]:
from openai import OpenAI

client = OpenAI()

system_message1 = """You are a business analyst building a goal model of stakeholder goals for a software application.
Goal models are directed, acyclic graphs in which edges trace from high-level goals to low-level goals through
refinement relationships. High-level goals describe what states stakeholders want to achieve, maintain or avoid in the
system. Low-level goals describe *how* the system will satsify high-level goals, tend to be more specific and describe
how the system will operate. High-level goals describe *why* the system aims to satisfy low-level goals, tend to be
more generic and describe what the stakeholder aims to accomplish independent of a specific software application."
"""

system_message2 = """You are a business analyst building a goal model of stakeholder goals for a software application.
To make sure that a software solution correctly solves a particular problem, we must first correctly understand and
define what problem needs to be solved. The aim of a software project is to improve the world by building some machine
expected to solve the particular problem. The machine consists of software to be developed and installed on some
computer platform.

The word *system* will be used to denote a set of components interacting with each other to satisfy some global
objectives.

The *system-as-is* is the system as it exists before the machine is built into it.
The *system-to-be* is the system as it should be when the machine will be built and operated in it.

Descriptive statements state properties about the system that hold regardless of how the system behaves. Such
properties hold typically because of some natural law or physical constraint. Descriptive statements are in
the indicative mood. The following statements are descriptive:
* If train doors are open, they are closed.
* A person cannot physically attend two meetings on different continents on the same day.
* The same book copy cannot be borrowed by two different people at the same time.

Prescriptive statements state desirable properties about the system that may hold or not depending on how the
system behaves. Such statements need to be enforced by system components. They are in the optative mood. The
following statements are prescriptive:
* Train doors shall always remain closed when the train is moving.
* A patron may not borrow more than three books at the same time.
* The meeting date must fit the constraints of all important participants.

We may need to negotiate, weaken, change, or find alternatives to prescriptive statements. However, we cannot
negotiate, weaken, change, or find alternatives to descriptive statements.

A goal is a prescriptive statement of intent that the system should satisfy through the cooperation of its
agents.

A goal is either a behavioral goal or a soft goal. A behavioral goal is either an Achieve goal or a Maintain goal.

Behavioral goals prescribe intended system behaviors declaritively. A behavioral goal implicitly defines a
maximal set of admissible system behaviors.

An Achieve goal prescribes intended behaviors where a target condition must sooner or later hold whenver some
other condition holds in the current system state. We prefix the goal name with the corresponding keyword. For
an Achieve goal we will write: Achieve[TargetCondition]. The corresponding specification has the following
informal temporal pattern:

Achieve[TargetCondition]:
[if CurrentCondition then] **sooner-or-later** TargetCondition.

Here is an example Achieve goal.
Achieve[MeetingRequestSatisfied]:
if a meeting is requested then **sooner-or-later** the meeting takes place and is attended by all imported invited participants.

Achieve[BookRequestSatisfied]:
if a book is requested then **sooner-or-later** a copy of the book is borrowed by the requesting patron.

Achieve[TrainProgress]:
if a train is at some platform then **sooner-or-later** the train is at the next platform.

A Maintain goal prescribes intended behaviors where a 'good' condition must always hold. Its specification takes the following informal temporal pattern:
Maintain[GoodCondition]:
[if CurrentCondition then] **always** GoodCondition, **always** (if someCondition **then** GoodCondition).

A Maintain goal may also be denoted by Avoid[BadCondition] for the dual variant of this pattern.
Avoid[BadCondition]:
[if CurrentCondition then] **always** **not** BadCondition

Here are a few example Maintain and Avoid goals.
Avoid[ParticipantConstraintsDisclosed]:
**always** **not** participant constraints disclosed to other invited participants.

Maintain[AccurateBookClassification]:
if a book is registered in the library directory then **always** its keyword-based classification is accurate."""

def prompt_model(prompt, system_message = system_message1):
    response = client.chat.completions.create(
      model="gpt-5.2-2025-12-11",
      #model="gpt-4o-2024-08-06",
      messages=[
        {
          "role": "system",
          "content": system_message
        },
        {
          "role": "user",
          "content": prompt
        }
      ]
    )
    return response.choices[0].message.content

In [2]:
import json
import os

data_path = 'data1_gpt52'

In [3]:
transcripts = json.load(open(os.path.join(data_path, "transcripts.json")))

In [4]:
def goal_statement_from_goalclusters(cluster, system_message=system_message1):
    """Develop a single goal statement from a goal cluster
    :param list(str) cluster: The list of goal statements in the goal cluster from which we derive a single goal statement.
    """
    prompt_prefix = "What is one goal statement that would describe the following goal statements?\n- "
    prompt_body = "\n- ".join(cluster)
    prompt_suffix = "\n\n"
    prompt = prompt_prefix + prompt_body + prompt_suffix
    statement = prompt_model(prompt, system_message)
    return statement

In [5]:
def goal_graph_from_goal_statements2(statements, system_message=system_message2):
    """Develop a goal graph from goal statements
    :param list(str) statements: The list of goal statements to generate a goal graph.
    """
    prompt_prefix = """A **goal model** shows how the system's functional and non-functional goals contribute to each other through
refinement links down to software requirements and environment assumptions. Graphically, a goal model is represented
by an AND/OR graph called a goal diagram. The goal nodes in such diagrams are annotated by their features and connected
through various types of edges. **Refinement** links indicate how a goal is AND-decomposed into conjoined sub-goals.
The same goal can be the target of multiple such links; each of them indicates an alternative way of refining the goal
into sub-goals. Leaf goals along refinement branches represent software requirements or environment assumptions needed
to enforce parent goals. In a goal diagram, **conflict** links may interconnect goal nodes to capture potential conflicts
among them.

The core of the goal model consists of a refinement graph showing how higher-level goals are refined into lower-level goals
and conversely, how lower-level goals contribute to higher-level goals. An AND-refinement link relates a goal to a set of
sub-goals. This set is called refinement of the parent goal. Each sub-goal in the refinement is said to **contribute**
to the parent goal. The meaning of an AND-refinement is that the parent goal can be satisfied by satisfying all subgoals
in the refinement.

The following code demonstrates a possible AND-refinement of the goal Achieve[BookRequestSatisfied] in a library system
which is refined into sub-goals Achieve[CopyBorrowedIfAvailable] and Achieve[CopyDueSoonForCheckOutIfNotAvailable].

```
achieve_copyborrowed_if_available = AchieveGoal(
    name="CopyBorrowedIfAvailable")

achieve_copyduesoonforcheckout_if_not_available = AchieveGoal(
    name="CopyDueSoonForCheckOutIfNotAvailable")

achieve_book_request_satisfied = AchieveGoal(
    name="BookRequestSatisfied",
    performs=None,
    refinements=[Refinement(
        complete=False,
        children=[
            achieve_copyborrowed_if_available,
            achieve_copyduesoonforcheckout_if_not_available]
    )])
```

    Following the code example above, generate code for a goal
    model with refinements where high-level goals are refined into low-level goals using the goals below: \n- """
    prompt_body = "\n- ".join(statements)
    prompt_suffix = "\n\n"
    prompt = prompt_prefix + prompt_body + prompt_suffix
    statement = prompt_model(prompt, system_message)
    return statement

In [2]:


clusters_27 = json.load(open(os.path.join(data_path, "cache", "clusters-27.json")))

In [3]:
clusters_27['0']

['Maintain global lounge access',
 'Enter personal lounge-access entitlements and keep them updated in one place',
 'Use a primary authoritative source to get accurate lounge-access information instead of relying on third-party information',
 'Keep information about what lounge access the speaker has up to date',
 "Determine what lounge access options are available based on the speaker's cards or passes",
 'Verify whether a card still provides lounge access at a specific airport',
 'Verify how many lounge visits remain for a limited-access card within a given period',
 'Manage lounge-access logistics across multiple cards and passes',
 'Confirm that a card still provides lounge access at a specific airport',
 'Verify that a card has remaining lounge access uses within a given time period',
 'Access all needed information in one place instead of visiting many different sites']

In [28]:
key = '0'
i = 0



In [31]:
print(prompt)

What is one goal statement that would describe the following goal statements?
- Maintain global lounge access
- Enter personal lounge-access entitlements and keep them updated in one place
- Use a primary authoritative source to get accurate lounge-access information instead of relying on third-party information
- Keep information about what lounge access the speaker has up to date
- Determine what lounge access options are available based on the speaker's cards or passes
- Verify whether a card still provides lounge access at a specific airport
- Verify how many lounge visits remain for a limited-access card within a given period
- Manage lounge-access logistics across multiple cards and passes
- Confirm that a card still provides lounge access at a specific airport
- Verify that a card has remaining lounge access uses within a given time period
- Access all needed information in one place instead of visiting many different sites




In [32]:
statement = prompt_model(prompt)

In [33]:
statement

'Maintain accurate, up-to-date, and centralized lounge-access entitlements and verification across all cards/passes and airports using an authoritative source.'

In [39]:
sorted_keys = sorted([int(i) for i in clusters_27.keys()])

In [42]:
# using system message 1
goal_statements = []
for i in sorted_keys:
    goal_statements.append(goal_statement_from_goalclusters(clusters_27[str(i)]))
    print(f"{i}...", "")
print("Done.")

0... 
1... 
2... 
3... 
4... 
5... 
6... 
7... 
8... 
9... 
10... 
11... 
12... 
13... 
14... 
15... 
16... 
17... 
18... 
19... 
20... 
21... 
22... 
23... 
24... 
25... 
26... 
27... 
28... 
29... 
30... 
31... 
32... 
33... 
34... 
35... 
36... 
37... 
38... 
39... 
40... 
41... 
42... 
43... 
44... 
45... 
46... 
47... 
Done.


In [49]:
goal_statement_from_goalclusters(clusters_27[str(1)])

'Achieve[FlightSearchResultsFilteredAndDateFlexible]: if a user performs a flight search and specifies filtering criteria (e.g., max flight duration, airline alliance/specific airline, departure airport within a city) and/or date-flexibility preferences (departure/return date ranges, including ±3 days or 7-day windows), then sooner-or-later the system returns flight results filtered accordingly and spanning the requested departure and return date ranges.'

In [46]:
i = 0
while i < len(goal_statements):
    print(f"{i} - {goal_statements[i]}")
    i += 1

0 - Maintain accurate, up-to-date global lounge-access entitlements and availability across all personal cards and passes using a single authoritative, centralized source.
1 - Enable users to search and filter flight options by flexible travel dates and preferred flight attributes (e.g., airline/alliance, departure airport, and maximum flight duration).
2 - Enable users to browse and purchase flights privately (anonymously), without logging in or being tracked via cookies.
3 - Plan and review a trip itinerary by entering a destination (country/region) and viewing total and return-journey timing details.
4 - Enable users to book flights (and optionally hotels) for themselves or others—using cash or rewards/points—at both advance and last‑minute timeframes to secure suitable prices/deals.
5 - **Improve the efficiency and effectiveness of research so users can quickly find accurate options, resolve mismatches, and understand the topic without a steep learning curve.**
6 - Find and select 

In [51]:
open(os.path.join(data_path, "clusters_27_goalgeneration_msg1.json"), "w").write(json.dumps(goal_statements))

7154

In [54]:
# using system message 2
goal_statements2 = []
for i in sorted_keys:
    goal_statements2.append(goal_statement_from_goalclusters(clusters_27[str(i)], system_message2))
    print(f"{i}...", end="")
print("Done.")

0...1...2...3...4...5...6...7...8...9...10...11...12...13...14...15...16...17...18...19...20...21...22...23...24...25...26...27...28...29...30...31...32...33...34...35...36...37...38...39...40...41...42...43...44...45...46...47...Done.


In [55]:
for statement in goal_statements2:
    print(statement)

Achieve[AccurateCentralizedLoungeAccessManagement]: if a speaker’s lounge-access entitlements (cards/passes) or travel context (airport/date) must be assessed, then sooner-or-later the system provides a single, up-to-date, authoritative view of (a) what lounge options are available and (b) whether each entitlement is valid at the specific airport and has remaining visits within the relevant time period.
Achieve[FlightSearchResultsMatchUserCriteria]:  
if a user performs a flight search with specified filters and date-range constraints (e.g., max duration, alliance/airline, departure airport within a city, and flexible departure/return date windows) then **sooner-or-later** the system returns a set of flight results filtered to satisfy all provided criteria.
Maintain[AnonymousFlightBrowsing]:
**always** a user can search for flights and browse flight options **without being identified**, i.e., **without logging in / being signed in** and **without cookies being stored or used to track t

In [56]:
open(os.path.join(data_path, "clusters_27_goalgeneration_msg2.json"), "w").write(json.dumps(goal_statements2))

14112

In [59]:
system_message3 = """You are a business analyst building a goal model of stakeholder goals for a software application.
To make sure that a software solution correctly solves a particular problem, we must first correctly understand and
define what problem needs to be solved. The aim of a software project is to improve the world by building some machine
expected to solve the particular problem. The machine consists of software to be developed and installed on some
computer platform.

The word *system* will be used to denote a set of components interacting with each other to satisfy some global
objectives.

The *system-as-is* is the system as it exists before the machine is built into it.
The *system-to-be* is the system as it should be when the machine will be built and operated in it.

Descriptive statements state properties about the system that hold regardless of how the system behaves. Such
properties hold typically because of some natural law or physical constraint. Descriptive statements are in
the indicative mood. The following statements are descriptive:
* If train doors are open, they are closed.
* A person cannot physically attend two meetings on different continents on the same day.
* The same book copy cannot be borrowed by two different people at the same time.

Prescriptive statements state desirable properties about the system that may hold or not depending on how the
system behaves. Such statements need to be enforced by system components. They are in the optative mood. The
following statements are prescriptive:
* Train doors shall always remain closed when the train is moving.
* A patron may not borrow more than three books at the same time.
* The meeting date must fit the constraints of all important participants.

We may need to negotiate, weaken, change, or find alternatives to prescriptive statements. However, we cannot
negotiate, weaken, change, or find alternatives to descriptive statements.

A goal is a prescriptive statement of intent that the system should satisfy through the cooperation of its
agents. A goal is either a behavioral goal or a soft goal. A behavioral goal is either an Achieve goal or a Maintain goal.

Behavioral goals prescribe intended system behaviors declaritively. A behavioral goal implicitly defines a
maximal set of admissible system behaviors.

An Achieve goal prescribes intended behaviors where a target condition must sooner or later hold whenver some
other condition holds in the current system state. We prefix the goal name with the corresponding keyword. For
an Achieve goal we will write: Achieve[TargetCondition]. The corresponding specification has the following
informal temporal pattern:

Achieve[TargetCondition]:
[if CurrentCondition then] **sooner-or-later** TargetCondition.

Here is an example Achieve goal.
Achieve[MeetingRequestSatisfied]:
if a meeting is requested then **sooner-or-later** the meeting takes place and is attended by all imported invited participants.

Achieve[BookRequestSatisfied]:
if a book is requested then **sooner-or-later** a copy of the book is borrowed by the requesting patron.

Achieve[TrainProgress]:
if a train is at some platform then **sooner-or-later** the train is at the next platform.

A Maintain goal prescribes intended behaviors where a 'good' condition must always hold. Its specification takes the following informal temporal pattern:
Maintain[GoodCondition]:
[if CurrentCondition then] **always** GoodCondition, **always** (if someCondition **then** GoodCondition).

A Maintain goal may also be denoted by Avoid[BadCondition] for the dual variant of this pattern.
Avoid[BadCondition]:
[if CurrentCondition then] **always** **not** BadCondition

Here are a few example Maintain and Avoid goals.
Avoid[ParticipantConstraintsDisclosed]:
**always** **not** participant constraints disclosed to other invited participants.

Maintain[AccurateBookClassification]:
if a book is registered in the library directory then **always** its keyword-based classification is accurate.

The **goal model** shows how the system's functional and non-functional goals contribute to each other through
refinement links down to software requirements and environment assumptions. Graphically, a goal model is represented
by an AND/OR graph called a goal diagram. The goal nodes in such diagrams are annotated by their features and connected
through various types of edges. **Refinement** links indicate how a goal is AND-decomposed into conjoined sub-goals.
The same goal can be the target of multiple such links; each of them indicates an alternative way of refining the goal
into sub-goals. Leaf goals along refinement branches represent software requirements or environment assumptions needed
to enforce parent goals. In a goal diagram, **conflict** links may interconnect goal nodes to capture potential conflicts
among them.

The core of the goal model consists of a refinement graph showing how higher-level goals are refined into lower-level goals
and conversely, how lower-level goals contribute to higher-level goals. An AND-refinement link relates a goal to a set of
sub-goals. This set is called refinement of the parent goal. Each sub-goal in the refinement is said to **contribute**
to the parent goal. The meaning of an AND-refinement is that the parent goal can be satisfied by satisfying all subgoals
in the refinement.

The following code demonstrates a possible AND-refinement of the goal Achieve[BookRequestSatisfied] in a library system
which is refined into sub-goals Achieve[CopyBorrowedIfAvailable] and Achieve[CopyDueSoonForCheckOutIfNotAvailable].

```
achieve_copyborrowed_if_available = AchieveGoal(
    name="CopyBorrowedIfAvailable")

achieve_copyduesoonforcheckout_if_not_available = AchieveGoal(
    name="CopyDueSoonForCheckOutIfNotAvailable")

achieve_book_request_satisfied = AchieveGoal(
    name="BookRequestSatisfied",
    performs=None,
    refinements=[Refinement(
        complete=False,
        children=[
            achieve_copyborrowed_if_available,
            achieve_copyduesoonforcheckout_if_not_available]
    )])
```
"""

In [60]:
"\n- ".join(goal_statements2)

'Achieve[AccurateCentralizedLoungeAccessManagement]: if a speaker’s lounge-access entitlements (cards/passes) or travel context (airport/date) must be assessed, then sooner-or-later the system provides a single, up-to-date, authoritative view of (a) what lounge options are available and (b) whether each entitlement is valid at the specific airport and has remaining visits within the relevant time period.\n- Achieve[FlightSearchResultsMatchUserCriteria]:  \nif a user performs a flight search with specified filters and date-range constraints (e.g., max duration, alliance/airline, departure airport within a city, and flexible departure/return date windows) then **sooner-or-later** the system returns a set of flight results filtered to satisfy all provided criteria.\n- Maintain[AnonymousFlightBrowsing]:\n**always** a user can search for flights and browse flight options **without being identified**, i.e., **without logging in / being signed in** and **without cookies being stored or used t

In [61]:
def goal_graph_from_goal_statements(statements, system_message=system_message3):
    """Develop a goal graph from goal statements
    :param list(str) statements: The list of goal statements to generate a goal graph.
    """
    prompt_prefix = "Generate a goal graph from the following goals?\n- "
    prompt_body = "\n- ".join(statements)
    prompt_suffix = "\n\n"
    prompt = prompt_prefix + prompt_body + prompt_suffix
    statement = prompt_model(prompt, system_message)
    return statement

In [63]:
goal_graph = goal_graph_from_goal_statements(goal_statements2, system_message3)

In [64]:
print(goal_graph)

```mermaid
graph TD

%% =========================
%% TOP-LEVEL OUTCOMES
%% =========================
TP(Achieve[TravelPlansSatisfied])
OTD(Achieve[OptimalTravelDealSelectedAndBooked])
TBC(Achieve[TravelBookingCompleted])
DBX(Achieve[DiverseTravelExperienceWithStopsAndLounges])
TS(Achieve[TimeSavedOnOptionSelectionAndTaskCompletion])

%% =========================
%% PRIVACY / TRUST (NFRs)
%% =========================
SGP(SoftGoal[Maximize personal privacy and trustworthy handling of personal data when using online services])
P0(Avoid[PrivacyExposureAndUnnecessaryAccountCreation])
P1(Maintain[AnonymousFlightBrowsing])
P2(Maintain[MinimizeLogin])

%% =========================
%% RESEARCH / KNOWLEDGE
%% =========================
RE0(Achieve[ResearchEfficiencyAndQualityImproved])
RE1(Achieve[EfficientAccurateResearchAndKnowledgeTransfer])
GUIDE(Achieve[UserCanEfficientlyFollowGuidedStepsToReachObjective])

%% =========================
%% END-TO-END TRAVEL PLANNING
%% =======================

In [67]:
goal_graph2 = goal_graph_from_goal_statements(goal_statements2, system_message2)

In [68]:
print(goal_graph2)

```mermaid
graph TD

%% =========================
%% TOP-LEVEL GOALS
%% =========================
G0[Achieve[TravelBookingCompleted]]
G1[Achieve[OptimalTravelDealSelectedAndBooked]]
G2[Achieve[TravelPlansSatisfied]]
G3[Achieve[DiverseTravelExperienceWithStopsAndLounges]]

P0[SoftGoal[Maximize personal privacy and trustworthy handling of personal data when using online services]]
P1[Avoid[PrivacyExposureAndUnnecessaryAccountCreation]]
P2[Maintain[AnonymousFlightBrowsing]]
P3[Maintain[MinimizeLogin]]

R0[Achieve[ResearchEfficiencyAndQualityImproved]]
R1[Achieve[EfficientAccurateResearchAndKnowledgeTransfer]]
R2[Achieve[TimeSavedOnOptionSelectionAndTaskCompletion]]
U0[Achieve[UserCanEfficientlyFollowGuidedStepsToReachObjective]]

%% High-level decompositions
G0 --> G4[Achieve[TripPlanned]]
G0 --> G5[Achieve[SelectSuitableFlightOption]]
G0 --> G6[Achieve[BookingCompletedOnProviderSite]]
G0 --> G7[Achieve[TravelOptionsFound]]
G0 --> G8[Achieve[TravelerCountsSpecified]]
G0 --> G9[Achieve[Tra

In [69]:
goal_graph3 = goal_graph_from_goal_statements2(goal_statements2, system_message2)
print(goal_graph3)

```python
# Goal model code with AND-refinements using the provided goals.
# (Follows the style of the example: goal nodes + Refinement(children=[...]))


# --- High-level goals ---------------------------------------------------------

achieve_travel_booking_completed = AchieveGoal(
    name="TravelBookingCompleted"
)

achieve_optimal_travel_deal_selected_and_booked = AchieveGoal(
    name="OptimalTravelDealSelectedAndBooked"
)

achieve_trip_planned = AchieveGoal(
    name="TripPlanned"
)

achieve_select_suitable_flight_option = AchieveGoal(
    name="SelectSuitableFlightOption"
)

achieve_best_travel_option_identified = AchieveGoal(
    name="BestTravelOptionIdentified"
)

achieve_accurate_centralized_lounge_access_management = AchieveGoal(
    name="AccurateCentralizedLoungeAccessManagement"
)

achieve_efficient_accurate_research_and_knowledge_transfer = AchieveGoal(
    name="EfficientAccurateResearchAndKnowledgeTransfer"
)

softgoal_maximize_privacy = SoftGoal(
    name="Maximize 

In [75]:
print(transcripts['goalmodel'][27])

from goalmodeling.schema import *

# OTA stands for online travel agency 
ota_agent = Agent(name="ExternalOTAs", agent_type=AgentType.ENVIRONMENT_AGENT)
# How/why goals related to location constraints 
departure_location = AchieveGoal(
    name="3 DisplayFlightsMatchingDepartureLocation<b>If</b>InputByUser", 
    annotation='How / why goal exploration')
arrival_location = AchieveGoal(
    name="4 DisplayFlightsMatchingArrivalLocation<b>If</b>InputByUser", 
    annotation='How / why goal exploration')
# How/why goals related to location constraints 
departure_date = AchieveGoal(
    name="6 DisplayFlightsMatchingDepartureDate<b>If</b>InputByUser", 
    annotation='How / why goal exploration')
arrival_date = AchieveGoal(
    name="7 DisplayFlightsMatchingArrivalDate<b>If</b>InputByUser", 
    annotation='How / why goal exploration')
time_of_departure_range = AchieveGoal(
    name="8 FilterFlightsByDepartureTimeRange<b>If</b>InputByUser", 
    annotation='How / why goal exploration')
time

In [76]:
clusters_15 = json.load(open(os.path.join(data_path, "cache", "clusters-15.json")))

In [77]:
sorted_keys_15 = sorted([int(i) for i in clusters_15.keys()])

In [79]:
goal_statements_15 = []
for i in sorted_keys_15:
    goal_statements_15.append(goal_statement_from_goalclusters(clusters_15[str(i)]))
    print(f"{i}...", end="")
print("Done.")

0...1...2...3...4...5...6...7...8...9...10...11...12...13...14...Done.


In [80]:
open(os.path.join(data_path, "clusters_15_goalgeneration_msg1.json"), "w").write(json.dumps(goal_statements_15))

1751

In [81]:
goal_statements2_15 = []
for i in sorted_keys_15:
    goal_statements2_15.append(goal_statement_from_goalclusters(clusters_15[str(i)], system_message2))
    print(f"{i}...", end="")
print("Done.")

0...1...2...3...4...5...6...7...8...9...10...11...12...13...14...Done.


In [82]:
goal_graph3_15 = goal_graph_from_goal_statements2(goal_statements2_15, system_message2)
print(goal_graph3_15)

```python
# Goal model for a cross-platform, low-ad, notification-bounded weather assistant

# --- Leaf/atomic goals (given) ---

achieve_weather_information_obtained_and_trusted = AchieveGoal(
    name="WeatherInformationObtainedAndTrusted"
)

achieve_weather_condition_notifications_received = AchieveGoal(
    name="WeatherConditionNotificationsReceived"
)

achieve_weather_information_obtained_before_going_out = AchieveGoal(
    name="WeatherInformationObtainedBeforeGoingOut"
)

achieve_weather_informed_outdoor_activity_plan_created = AchieveGoal(
    name="Weather-Informed Outdoor Activity Plan Created"
)

avoid_unwanted_advertisements_displayed = MaintainGoal(   # Avoid[...] is dual of Maintain
    name="UnwantedAdvertisementsDisplayed"
)

achieve_cross_platform_consistency_established = AchieveGoal(
    name="CrossPlatformConsistencyEstablished"
)

achieve_weather_forecast_reviewed_regularly_for_planning = AchieveGoal(
    name="WeatherForecastReviewedRegularlyForPlanning"
)

achie

In [83]:
print(transcripts['goalmodel'][15])

from goalmodeling.schema import *

def weather_goal_model(host="https://mermaid.live"):
    # Agents
    user_agent = Agent("User", AgentType.ENVIRONMENT_AGENT, annotation="User performing weather checks")
    agent_user1 = Agent("User", AgentType.ENVIRONMENT_AGENT)  # Separate user agent for enabling voice assistant
    agent_user2 = Agent("User", AgentType.ENVIRONMENT_AGENT)  # Separate user agent for checking device notifications
    system_agent_hail = Agent("System", AgentType.SOFTWARE_AGENT)
    system_agent_tornado = Agent("System", AgentType.SOFTWARE_AGENT)
    system_agent_drought = Agent("System", AgentType.SOFTWARE_AGENT)

    # Operations for the agent actions
    enable_voice_assistant_operation = Operation("Enable Voice Assistant", OperationCategory.ENVIRONMENT_OPERATION)
    check_notifications_operation = Operation("Check Device Notifications", OperationCategory.ENVIRONMENT_OPERATION)
    send_hail_update_operation = Operation("Publish Hail Update", OperationCategory.SO

In [6]:
clusters = {}
sorted_keys = {}
goal_statements = {}
goal_statements2 = {}

In [7]:
goal_graph3 = {}

In [8]:
key = 27

In [9]:
clusters[key] = json.load(open(os.path.join(data_path, "cache", f"clusters-{key}.json")))

In [10]:
sorted_keys[key] = sorted([int(i) for i in clusters[key].keys()])

In [11]:
goal_statements[key] = json.load(open(os.path.join(data_path, f"clusters_{key}_goalgeneration_msg1.json")))

In [14]:
goal_statements[key] = []

In [15]:
for cluster in clusters[27]:
    goal_statements[27].extend(clusters[27][cluster])

In [17]:
len(goal_statements[27])

362

In [29]:
if key not in goal_statements:
    goal_statements[key] = []
    for i in sorted_keys[key]:
        goal_statements[key].append(goal_statement_from_goalclusters(clusters[key][str(i)]))
        print(f"{i}...", end="")
        print("Done.")
    open(os.path.join(data_path, f"clusters_{key}_goalgeneration_msg1.json"), "w").write(json.dumps(goal_statements[key]))

In [19]:
goal_statements2[key] = []

In [20]:
for statement in goal_statements[key]:
    goal_statements2[key].append(goal_statement_from_goalclusters(statement, system_message2))
    print(f"{i}...", end="")
print("Done.")

47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...47...

In [21]:
open(os.path.join(data_path, f"clusters_{key}_goalgeneration_msg2_all_statements.json"), "w").write(json.dumps(goal_statements2[key]))

70592

In [22]:
goal_graph3[key] = goal_graph_from_goal_statements2(goal_statements2[key], system_message2)

In [23]:
open(os.path.join(data_path, f"goalgraph_msg2_all_statements{key}.py"), "w").write(goal_graph3[key])

15180

In [41]:
# dump the interviewer's goal model
open(os.path.join(data_path, f"goalgraph_transcript_{key}.py"), "w").write(transcripts['goalmodel'][key])

0

In [43]:
transcripts['goalmodel'][key]

''

In [83]:
def generate_goal_graph(k):
    key = k
    clusters[key] = json.load(open(os.path.join(data_path, "cache", f"clusters-{key}.json")))
    sorted_keys[key] = sorted([int(i) for i in clusters[key].keys()])
    print(f"Loading goal_statements[{key}]...")
    try:
        goal_statements[key] = json.load(open(os.path.join(data_path, f"clusters_{key}_goalgeneration_msg1.json"), encoding="utf-8"))
    except FileNotFoundError:
        if key not in goal_statements:
            goal_statements[key] = []
        for i in sorted_keys[key]:
            goal_statements[key].append(goal_statement_from_goalclusters(clusters[key][str(i)]))
            print(f"{i}...", end="")
        open(os.path.join(data_path, f"clusters_{key}_goalgeneration_msg1.json"), "w", encoding="utf-8").write(json.dumps(goal_statements[key]))
    print("Done.")


    print(f"Loading goal_statements2[{key}]...")
    try:
        goal_statements2[key] = json.load(open(os.path.join(data_path, f"clusters_{key}_goalgeneration_msg2.json"), encoding="utf-8"))
    except FileNotFoundError:
        if key not in goal_statements2:
            goal_statements2[key] = []
            for i in sorted_keys[key]:
                goal_statements2[key].append(goal_statement_from_goalclusters(clusters[key][str(i)], system_message2))
                print(f"{i}...", end="")
            open(os.path.join(data_path, f"clusters_{key}_goalgeneration_msg2.json"), "w", encoding="utf-8").write(json.dumps(goal_statements2[key]))
    print("Done.")


    print(f"Loading goal_graph3[{key}]...")
    try:
        goal_graph3[key] = open(os.path.join(data_path, f"goalgraph_msg2_{key}.py"), encoding="utf-8").read()
    except FileNotFoundError:
        if key not in goal_graph3:
            goal_graph3[key] = goal_graph_from_goal_statements2(goal_statements2[key], system_message2)
            open(os.path.join(data_path, f"goalgraph_msg2_{key}.py"), "w", encoding="utf-8").write(goal_graph3[key])
    print("Done")

    # dump the interviewer's goal model
    open(os.path.join(data_path, f"goalgraph_transcript_{key}.py"), "w", encoding="utf-8").write(transcripts['goalmodel'][key])

In [84]:
generate_goal_graph(7)

Loading goal_statements[7]...
Done.
Loading goal_statements2[7]...
Done.
Loading goal_graph3[7]...
Done


In [85]:
generate_goal_graph(5)

Loading goal_statements[5]...
0...1...2...3...4...5...6...7...8...9...10...11...12...Done.
Loading goal_statements2[5]...
0...1...2...3...4...5...6...7...8...9...10...11...12...Done.
Loading goal_graph3[5]...
Done


In [86]:
generate_goal_graph(9)

Loading goal_statements[9]...
0...1...2...3...4...5...6...7...8...9...10...11...12...13...14...15...16...17...Done.
Loading goal_statements2[9]...
0...1...2...3...4...5...6...7...8...9...10...11...12...13...14...15...16...17...Done.
Loading goal_graph3[9]...
Done


In [87]:
generate_goal_graph(10)

Loading goal_statements[10]...
0...1...2...3...4...5...6...7...8...9...10...11...12...13...14...Done.
Loading goal_statements2[10]...
0...1...2...3...4...5...6...7...8...9...10...11...12...13...14...Done.
Loading goal_graph3[10]...
Done
